# eGeMAPS Baseline for Parkinson's Disease Classification
## XGBoost classification on the NeuroVoz vowels

Baseline for comparison with the eaQHM AM-FM features. The 88 eGeMAPSv02 functionals, pre-extracted with openSMILE from the 16 kHz NeuroVoz vowel recordings, are loaded from a CSV file and classified with XGBoost (`scale_pos_weight` set to the speaker-level HC/PD ratio). Feature columns that are zero for every recording are removed.

The evaluation protocol is identical to the other NeuroVoz notebooks: the same predefined speaker-independent folds (`master_cv_folds_neurovoz_vowels.csv`, 5 repeats × 10 outer folds, stratified by label and gender), a 5-fold inner grid search over the number of trees, learning rate, maximum depth, row subsampling and column subsampling (ROC AUC), and results at sample and speaker level.

### Imports

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score,
                             log_loss, brier_score_loss)
from typing import Tuple

warnings.filterwarnings("ignore")

### 0. Configuration

In [2]:
# ==========================================
# 0. CONFIGURATION
# ==========================================
task              = "neurovoz_vowels"
base_random_state = 42
N_INNER_SPLITS    = 5

bad_files = [
    'HC_A1_0084', 'HC_A1_0087', 'HC_A2_0068', 'HC_A2_0071',
    'HC_A2_0084', 'HC_A2_0087', 'HC_E2_0071', 'HC_E2_0087',
    'HC_E3_0034', 'HC_I1_0071', 'HC_I1_0084', 'HC_I2_0049',
    'HC_I2_0063', 'HC_I2_0083', 'HC_I2_0084', 'HC_I2_0087',
    'HC_O1_0071', 'HC_O1_0083', 'HC_O1_0084', 'HC_O2_0071',
    'HC_O2_0087', 'HC_U1_0053', 'HC_U1_0071', 'HC_U1_0083',
    'HC_U1_0084', 'HC_U1_0087', 'HC_U2_0144', 'PD_A1_0078',
    'PD_E2_0014', 'PD_E2_0018', 'PD_E2_0022', 'PD_E2_0023',
    'PD_E2_0024', 'PD_E2_0077', 'PD_E2_0078', 'PD_E2_0079',
    'PD_E3_0004', 'PD_E3_0007', 'PD_E3_0008', 'PD_E3_0011',
    'PD_E3_0012', 'PD_O1_0007', 'PD_O1_0009', 'PD_O1_0010',
    'PD_O1_0014', 'PD_O1_0015', 'PD_O1_0070',
]

### 1. Load pre-extracted eGeMAPS features

In [3]:
# ==========================================
# 1. LOAD PRE-EXTRACTED FEATURES
# ==========================================
features_path = "../features/neurovoz_vowels_egemapsv02_16k_5ms.csv"

if not os.path.exists(features_path):
    raise FileNotFoundError(f"Feature file not found: {features_path}")

print(f"Loading features from {features_path}...")
data = pd.read_csv(features_path)
print(f"Loaded. Shape: {data.shape}")

n_before = len(data)
data = data[~data['name'].isin(bad_files)].reset_index(drop=True)
print(f"Removed {n_before - len(data)} bad files. Remaining: {len(data)}")

Loading features from ../features/neurovoz_vowels_egemapsv02_16k_5ms.csv...
Loaded. Shape: (1019, 93)
Removed 0 bad files. Remaining: 1019


### 2. Data cleaning and class-balance weight

In [4]:
# ==========================================
# 2. DATA CLEANING
# ==========================================
print("\nClass distribution:");      print(data['label'].value_counts())
print("Gender distribution (0=F, 1=M):"); print(data[['speaker','gender']].drop_duplicates()['gender'].value_counts())
print(f"Total NaNs: {data.isna().sum().sum()}")

assert data['label'].isin([0, 1]).all(),   "Unexpected label values"
assert data['gender'].isin([0, 1]).all(),  "Unexpected gender values — check the gender metadata used at extraction"

# Class balance weight for XGBoost (speaker-level) 
n_neg            = (data[['speaker','label']].drop_duplicates()['label'] == 0).sum()
n_pos            = (data[['speaker','label']].drop_duplicates()['label'] == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f"\nSpeaker-level HC/PD ratio: {scale_pos_weight:.4f} → scale_pos_weight")


Class distribution:
label
1    545
0    474
Name: count, dtype: int64
Gender distribution (0=F, 1=M):
gender
1    61
0    50
Name: count, dtype: int64
Total NaNs: 0

Speaker-level HC/PD ratio: 1.0943 → scale_pos_weight


### 3. Load folds and merge

In [6]:
# ==========================================
# 3. LOAD FOLDS AND MERGE
# ==========================================
fold_file = f"../folds/master_cv_folds_{task}.csv"
print(f"\nLoading predefined folds from {fold_file}...")
fold_map = pd.read_csv(fold_file)
fold_map['speaker'] = fold_map['speaker'].astype(str).str.strip().str.zfill(4)

repeat_fold_cols = [c for c in fold_map.columns if c.startswith("Repeat_") and c.endswith("_Fold")]
speaker_folds    = fold_map[['speaker'] + repeat_fold_cols].drop_duplicates(subset='speaker')

data['speaker'] = data['speaker'].astype(str).str.strip().str.zfill(4)
data = data.merge(speaker_folds, on='speaker', how='inner').reset_index(drop=True)

if data.empty:
    raise ValueError("CRITICAL: DataFrame empty after fold merge — check speaker ID format")

print(f"Shape after fold merge: {data.shape}")

for col in repeat_fold_cols:
    assert data.groupby('speaker')[col].nunique().max() == 1, \
        f"Speaker has inconsistent fold assignments in {col}"
print("✅ All speakers have consistent fold assignments")

original_speakers = set(fold_map['speaker'].str.strip().str.zfill(4))
current_speakers  = set(data['speaker'].str.strip().str.zfill(4))
missing = current_speakers - original_speakers
if missing:
    print(f"⚠️  {len(missing)} speakers not in fold file: {missing}")


Loading predefined folds from ../folds/master_cv_folds_neurovoz_vowels.csv...
Shape after fold merge: (1019, 98)
✅ All speakers have consistent fold assignments


### 4. Features and groups

In [7]:
# ==========================================
# 4. FINALISE FEATURES AND GROUPS
# ==========================================
groups = data['speaker'].values

data['stratify_key'] = data['label'].astype(str) + "_" + data['gender'].astype(str)

cols_to_drop = ['name', 'speaker', 'vowel', 'label', 'gender',
                'stratify_key', 'sample_id'] + repeat_fold_cols
X = data.drop(columns=cols_to_drop, errors='ignore')
y          = data['label']
y_stratify = data['stratify_key']

N_REPEATS      = len(repeat_fold_cols)
N_OUTER_SPLITS = data[repeat_fold_cols[0]].nunique()

print(f"\nFeature matrix : {X.shape}")
print(f"Unique speakers: {len(np.unique(groups))}")
print(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS}")
print("Stratify key counts:"); print(y_stratify.value_counts().sort_index())


Feature matrix : (1019, 88)
Unique speakers: 111
Repeats x Folds: 5 x 10
Stratify key counts:
stratify_key
0_0    230
0_1    244
1_0    216
1_1    329
Name: count, dtype: int64


### 4a. Remove all-zero feature columns

In [8]:
# Remove all-zero columns 
zero_cols = [c for c in X.columns if (X[c] == 0).all()]
if zero_cols:
    print(f"\n⚠️  Dropping {len(zero_cols)} all-zero columns: {zero_cols}")
    X = X.drop(columns=zero_cols)
else:
    print("\n✅ No all-zero columns found.")
print(f"Feature matrix after zero-column removal: {X.shape}")


✅ No all-zero columns found.
Feature matrix after zero-column removal: (1019, 88)


### 5. Helper functions

In [9]:
# ==========================================
# 5. HELPERS
# ==========================================
def aggregate_mean_by_group(
    y: np.ndarray, p: np.ndarray, g: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    y, p, g = np.asarray(y).astype(int), np.asarray(p).astype(float), np.asarray(g)
    uniq = np.unique(g)
    y_g  = np.zeros(len(uniq), dtype=int)
    p_g  = np.zeros(len(uniq), dtype=float)
    for i, gg in enumerate(uniq):
        idx    = np.where(g == gg)[0]
        p_g[i] = float(np.mean(p[idx])) if len(idx) else float('nan')
        y_g[i] = int(np.mean(y[idx]) >= 0.5) if len(idx) else 0
    return y_g, p_g, uniq

### 6. Result storage

In [10]:
# ==========================================
# 6. RESULT STORAGE
# ==========================================
metrics_keys = ["accuracy", "f1", "auc", "precision", "recall", "log_loss", "brier"]
results_sample  = {k: [] for k in metrics_keys}
results_speaker = {k: [] for k in metrics_keys}
conf_matrices_sample  = []
conf_matrices_speaker = []

### 7. Repeated nested cross-validation (slow: 324-point grid)

In [ ]:
# ==========================================
# 7. MAIN LOOP
# ==========================================
print(f"\nStarting XGBoost Repeated Nested CV ({N_REPEATS} Repeats × {N_OUTER_SPLITS} Folds)…")

for repeat_idx, repeat_col in enumerate(repeat_fold_cols):
    repeat_num   = repeat_idx + 1
    current_seed = base_random_state + repeat_idx

    print(f"\n{'='*80}")
    print(f"REPEAT {repeat_num}/{N_REPEATS}  (Seed: {current_seed} | Column: {repeat_col})")
    print("="*80)

    for fold_id in range(N_OUTER_SPLITS):

        test_mask  = (data[repeat_col] == fold_id).values
        train_mask = ~test_mask

        X_train, X_test = X.iloc[train_mask], X.iloc[test_mask]
        y_train, y_test = y.iloc[train_mask], y.iloc[test_mask]
        g_train_np      = groups[train_mask]
        g_test_np       = groups[test_mask]
        y_strat_train   = y_stratify.iloc[train_mask]

        # LEAKAGE CHECKS
        assert len(set(g_train_np) & set(g_test_np)) == 0, \
            f"❌ OUTER LEAKAGE R{repeat_num} Fold {fold_id+1}"
        n_spk_train = len(set(g_train_np))
        n_spk_test  = len(set(g_test_np))
        assert n_spk_train + n_spk_test == len(np.unique(groups)), \
            f"❌ Speaker count mismatch R{repeat_num} Fold {fold_id+1}"
        assert len(set(np.where(train_mask)[0]) & set(np.where(test_mask)[0])) == 0, \
            f"❌ SAMPLE OVERLAP R{repeat_num} Fold {fold_id+1}"
        forbidden = ["label", "speaker", "gender", "stratify_key", "name", "vowel"]
        assert not any(c in X_train.columns for c in forbidden), \
            "❌ Forbidden columns in features"

        inner_cv = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True,
            random_state=current_seed + fold_id,
        )
        for i, (tr_i, va_i) in enumerate(inner_cv.split(X_train, y_strat_train, g_train_np)):
            assert len(set(g_train_np[tr_i]) & set(g_train_np[va_i])) == 0, \
                f"❌ INNER LEAKAGE R{repeat_num} Fold {fold_id+1} Inner {i+1}"

        print(f"R{repeat_num} Fold {fold_id+1:2d}: ✅ All checks passed | "
              f"Train: {len(X_train)} samples ({n_spk_train} spk) | "
              f"Test: {len(X_test)} samples ({n_spk_test} spk)")

        # GridSearchCV 
        xgb = XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            scale_pos_weight=scale_pos_weight,
            random_state=current_seed,
            n_jobs=-1,
        )
        param_grid = {
            "n_estimators":     [100, 200, 300, 500],
            "learning_rate":    [0.01, 0.05, 0.1],
            "max_depth":        [3, 5, 7],
            "subsample":        [0.5, 0.8, 1.0],
            "colsample_bytree": [0.5, 0.8, 1.0],
        }

        grid_search = GridSearchCV(
            estimator=xgb,
            param_grid=param_grid,
            cv=inner_cv.split(X_train, y_strat_train, g_train_np),
            scoring="roc_auc",
            n_jobs=-1,
            verbose=0,
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        bp         = grid_search.best_params_

        # Sample-level predictions 
        y_pred_sample  = best_model.predict(X_test)
        y_proba_sample = best_model.predict_proba(X_test)[:, 1]

        results_sample["accuracy"].append(  accuracy_score(  y_test, y_pred_sample))
        results_sample["f1"].append(        f1_score(        y_test, y_pred_sample, zero_division=0))
        results_sample["auc"].append(       roc_auc_score(   y_test, y_proba_sample))
        results_sample["precision"].append( precision_score( y_test, y_pred_sample, zero_division=0))
        results_sample["recall"].append(    recall_score(    y_test, y_pred_sample, zero_division=0))
        results_sample["log_loss"].append(  log_loss(        y_test, y_proba_sample))
        results_sample["brier"].append(     brier_score_loss(y_test, y_proba_sample))
        conf_matrices_sample.append(confusion_matrix(y_test, y_pred_sample))

        # Speaker-level predictions 
        y_test_spk, y_proba_spk, _ = aggregate_mean_by_group(
            y_test.values, y_proba_sample, g_test_np
        )
        y_pred_spk = (y_proba_spk >= 0.5).astype(int)

        results_speaker["accuracy"].append(  accuracy_score(  y_test_spk, y_pred_spk))
        results_speaker["f1"].append(        f1_score(        y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["auc"].append(       roc_auc_score(   y_test_spk, y_proba_spk) if len(np.unique(y_test_spk)) > 1 else np.nan)
        results_speaker["precision"].append( precision_score( y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["recall"].append(    recall_score(    y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["log_loss"].append(  log_loss(        y_test_spk, y_proba_spk) if len(np.unique(y_test_spk)) > 1 else np.nan)
        results_speaker["brier"].append(     brier_score_loss(y_test_spk, y_proba_spk))
        conf_matrices_speaker.append(confusion_matrix(y_test_spk, y_pred_spk))

        print(f"           Est={bp['n_estimators']}, LR={bp['learning_rate']}, "
              f"Depth={bp['max_depth']}, Sub={bp['subsample']}, ColSample={bp['colsample_bytree']}")
        print(f"           [Sample]  Acc: {results_sample['accuracy'][-1]:.4f} | "
              f"F1: {results_sample['f1'][-1]:.4f} | AUC: {results_sample['auc'][-1]:.4f}")
        print(f"           [Speaker] Acc: {results_speaker['accuracy'][-1]:.4f} | "
              f"F1: {results_speaker['f1'][-1]:.4f} | AUC: {results_speaker['auc'][-1]:.4f}")
        print("-" * 60)

### 8. Summary and confusion matrices

In [ ]:
# ==========================================
# 8. SUMMARY
# ==========================================
total_folds = N_REPEATS * N_OUTER_SPLITS
print(f"\n{'='*55}")
print(f"FINAL XGBoost RESULTS ({total_folds} Total Folds)")
print("="*55)
print(f"{'Metric':<12} | {'Sample-Level':<25} | {'Speaker-Level':<25}")
print("-" * 68)

for metric in metrics_keys:
    ms = np.nanmean(results_sample[metric]);  ss = np.nanstd(results_sample[metric])
    mk = np.nanmean(results_speaker[metric]); sk = np.nanstd(results_speaker[metric])
    print(f"{metric.capitalize():<12} | {ms:.4f} ± {ss:.4f}            | {mk:.4f} ± {sk:.4f}")

print("\n--- Aggregated Confusion Matrix (Sample-Level) ---")
print(np.sum(conf_matrices_sample, axis=0))
print("\n--- Aggregated Confusion Matrix (Speaker-Level) ---")
print(np.sum(conf_matrices_speaker, axis=0))